In [1]:
import pandas as pd
import numpy as np


In [4]:
df = pd.read_csv("/content/cleaned_deliveries.csv")


In [5]:
df.head()


,match_id,inning,over_ball,over,ball,batting_team,bowling_team,batter,non_striker,bowler,runs_off_bat,extras,iswide,isnoball,dismissal_kind,player_dismissed,date,is_wicket
0,335982,1,0.1,0,1,Kolkata Knight Riders,RCB,SC Ganguly,BB McCullum,P Kumar,0,1,0,0,Not Out,NaN,2008-04-18,0
1,335982,1,0.2,0,2,Kolkata Knight Riders,RCB,BB McCullum,SC Ganguly,P Kumar,0,0,0,0,Not Out,NaN,2008-04-18,0
2,335982,1,0.3,0,3,Kolkata Knight Riders,RCB,BB McCullum,SC Ganguly,P Kumar,0,1,1,0,Not Out,NaN,2008-04-18,0
3,335982,1,0.4,0,4,Kolkata Knight Riders,RCB,BB McCullum,SC Ganguly,P Kumar,0,0,0,0,Not Out,NaN,2008-04-18,0
4,335982,1,0.5,0,5,Kolkata Knight Riders,RCB,BB McCullum,SC Ganguly,P Kumar,0,0,0,0,Not Out,NaN,2008-04-18,0


In [6]:
df = df.sort_values(
    by=["batter", "date", "match_id", "inning", "over_ball"]
).reset_index(drop=True)


In [7]:
batsman_match = (
    df.groupby(["match_id", "batter"])
    .agg(
        runs=("runs_off_bat", "sum"),
        balls=("runs_off_bat", "count")
    )
    .reset_index()
)


In [8]:
batsman_match.head()


,match_id,batter,runs,balls
0,335982,AA Noffke,9,12
1,335982,B Akhil,0,2
2,335982,BB McCullum,158,77
3,335982,CL White,6,10
4,335982,DJ Hussey,12,12


In [9]:
batsman_match["strike_rate"] = (
    batsman_match["runs"] / batsman_match["balls"]
) * 100


In [10]:
batsman_match["career_avg_runs"] = (
    batsman_match
    .groupby("batter")["runs"]
    .expanding()
    .mean()
    .reset_index(level=0, drop=True)
)


In [11]:
batsman_match["avg_runs_last5"] = (
    batsman_match
    .groupby("batter")["runs"]
    .rolling(5)
    .mean()
    .reset_index(level=0, drop=True)
)


In [12]:
batsman_match["avg_runs_last10"] = (
    batsman_match
    .groupby("batter")["runs"]
    .rolling(10)
    .mean()
    .reset_index(level=0, drop=True)
)


In [13]:
batsman_match["avg_runs_last5"] = batsman_match["avg_runs_last5"].fillna(
    batsman_match["career_avg_runs"]
)

batsman_match["avg_runs_last10"] = batsman_match["avg_runs_last10"].fillna(
    batsman_match["career_avg_runs"]
)


In [14]:
batsman_features = batsman_match[
    [
        "match_id",
        "batter",
        "avg_runs_last5",
        "avg_runs_last10",
        "career_avg_runs",
        "strike_rate",
        "runs"
    ]
]


In [15]:
batsman_features.head()


,match_id,batter,avg_runs_last5,avg_runs_last10,career_avg_runs,strike_rate,runs
0,335982,AA Noffke,9.0,9.0,9.0,75.000000,9
1,335982,B Akhil,0.0,0.0,0.0,0.000000,0
2,335982,BB McCullum,158.0,158.0,158.0,205.194805,158
3,335982,CL White,6.0,6.0,6.0,60.000000,6
4,335982,DJ Hussey,12.0,12.0,12.0,100.000000,12


In [16]:
batsman_features.to_csv(
    "batsman_match_features.csv",
    index=False
)


In [17]:
print("Batsman features saved successfully!")


Batsman features saved successfully!


In [18]:
bowler_match = (
    df.groupby(["match_id", "bowler"])
    .agg(
        wickets=("is_wicket", "sum"),
        balls=("is_wicket", "count")
    )
    .reset_index()
)


In [19]:
bowler_match.head()


,match_id,bowler,wickets,balls
0,335982,AA Noffke,1,25
1,335982,AB Agarkar,3,28
2,335982,AB Dinda,2,20
3,335982,CL White,0,7
4,335982,I Sharma,1,19


In [20]:
bowler_match["career_avg_wickets"] = (
    bowler_match
    .groupby("bowler")["wickets"]
    .expanding()
    .mean()
    .reset_index(level=0, drop=True)
)


In [21]:
bowler_match["avg_wickets_last5"] = (
    bowler_match
    .groupby("bowler")["wickets"]
    .rolling(5)
    .mean()
    .reset_index(level=0, drop=True)
)


In [22]:
bowler_match["avg_wickets_last10"] = (
    bowler_match
    .groupby("bowler")["wickets"]
    .rolling(10)
    .mean()
    .reset_index(level=0, drop=True)
)


In [23]:
bowler_match["avg_wickets_last5"] = bowler_match["avg_wickets_last5"].fillna(
    bowler_match["career_avg_wickets"]
)

bowler_match["avg_wickets_last10"] = bowler_match["avg_wickets_last10"].fillna(
    bowler_match["career_avg_wickets"]
)


In [24]:
bowler_features = bowler_match[
    [
        "match_id",
        "bowler",
        "avg_wickets_last5",
        "avg_wickets_last10",
        "career_avg_wickets",
        "wickets"
    ]
]


In [25]:
bowler_features.head()


,match_id,bowler,avg_wickets_last5,avg_wickets_last10,career_avg_wickets,wickets
0,335982,AA Noffke,1.0,1.0,1.0,1
1,335982,AB Agarkar,3.0,3.0,3.0,3
2,335982,AB Dinda,2.0,2.0,2.0,2
3,335982,CL White,0.0,0.0,0.0,0
4,335982,I Sharma,1.0,1.0,1.0,1


In [26]:
bowler_features.to_csv(
    "bowler_match_features.csv",
    index=False
)


In [27]:
print("Bowler features saved successfully!")


Bowler features saved successfully!


In [28]:
print("Feature Engineering completed successfully!")


Feature Engineering completed successfully!
